# Prompt 10: Testing & Evaluation of Prompts

1. Render tests (template well-formedness)
2. Format tests (schema validity)
3. Golden-set regression with threshold
4. Snapshot test for stability
5. Parametrized pytest-style structure
6. Exercises: pin a prompt + tests to CI

Deps: `pip install pydantic jinja2 anthropic openai`

In [ ]:
import os, json, re
from jinja2 import Environment, BaseLoader
from pydantic import BaseModel, ValidationError

env = Environment(loader=BaseLoader(), trim_blocks=True, lstrip_blocks=True)

TMPL = env.from_string('''Extract JSON matching this schema:
{"vendor": string, "total_usd": number, "due": "YYYY-MM-DD" or null}

Input:
{{ text }}
Return JSON only.''')

class Invoice(BaseModel):
    vendor: str
    total_usd: float
    due: str | None = None

def call(system, user, max_tokens=300):
    if os.getenv('ANTHROPIC_API_KEY'):
        from anthropic import Anthropic
        r = Anthropic().messages.create(model='claude-sonnet-4-6', max_tokens=max_tokens, temperature=0,
            system=system, messages=[{'role':'user','content':user}])
        return r.content[0].text
    if os.getenv('OPENAI_API_KEY'):
        from openai import OpenAI
        r = OpenAI().chat.completions.create(model='gpt-4o-mini', max_tokens=max_tokens, temperature=0,
            messages=[{'role':'system','content':system},{'role':'user','content':user}])
        return r.choices[0].message.content
    return '{}'

## 1. Render Tests (Deterministic, Free)

In [ ]:
def test_render_produces_schema_and_input():
    p = TMPL.render(text='Sample invoice text')
    assert 'vendor' in p
    assert 'Sample invoice text' in p
    assert 'Return JSON only' in p
    print('PASS test_render_produces_schema_and_input')

def test_render_fails_on_missing_var():
    try:
        TMPL.render()
    except Exception as e:
        # Jinja2 renders missing vars as empty by default
        pass
    print('PASS test_render_fails_on_missing_var (or renders empty)')

test_render_produces_schema_and_input()
test_render_fails_on_missing_var()

## 2. Format Tests (Parse + Validate)

In [ ]:
GOLDEN = [
    {
        'id': 'basic',
        'input': 'Bill from NovaCore. Total: $438.00. Due: 2026-05-10.',
        'expected': {'vendor':'NovaCore','total_usd':438.0,'due':'2026-05-10'},
    },
    {
        'id': 'no_due',
        'input': 'Bill from Acme. Total: $1200. Please pay promptly.',
        'expected': {'vendor':'Acme','total_usd':1200.0,'due':None},
    },
    {
        'id': 'comma_amt',
        'input': 'Vendor: Globex.  Amount: $12,345.67   Due on 2026-06-01',
        'expected': {'vendor':'Globex','total_usd':12345.67,'due':'2026-06-01'},
    },
]

def extract_json(text):
    m = re.search(r'\{.*\}', text, re.S)
    return json.loads(m.group(0)) if m else None

def test_all_outputs_valid_schema():
    failures = []
    for ex in GOLDEN:
        out = call('You extract invoices. Return JSON only.', TMPL.render(text=ex['input']))
        obj = extract_json(out)
        try:
            Invoice(**(obj or {}))
        except (ValidationError, TypeError) as e:
            failures.append((ex['id'], str(e)[:80]))
    if failures:
        print('FAIL schema:', failures)
    else:
        print(f'PASS test_all_outputs_valid_schema ({len(GOLDEN)}/{len(GOLDEN)})')

test_all_outputs_valid_schema()

## 3. Golden-Set Regression with Threshold

In [ ]:
BASELINE_ACCURACY = 0.80   # pin this; adjust deliberately

def accuracy_on_golden():
    hits = 0
    for ex in GOLDEN:
        out = call('You extract invoices. Return JSON only.', TMPL.render(text=ex['input']))
        obj = extract_json(out) or {}
        # field-level partial credit
        ok = (obj.get('vendor','').lower() == ex['expected']['vendor'].lower()
              and abs(float(obj.get('total_usd', -1)) - ex['expected']['total_usd']) < 0.01)
        hits += int(ok)
    return hits / len(GOLDEN)

acc = accuracy_on_golden()
print(f'accuracy: {acc:.2f}  baseline: {BASELINE_ACCURACY}')
assert acc >= BASELINE_ACCURACY - 0.05, f'regression: {acc} < {BASELINE_ACCURACY - 0.05}'
print('PASS test_accuracy_above_threshold')

## 4. Snapshot for Stability

In [ ]:
# In real projects: store snapshots as files in tests/snapshots/*.txt; update with a CLI flag.

fixed_input = 'Bill from NovaCore. Total: $438.00. Due: 2026-05-10.'
snapshot_expected = {'vendor':'NovaCore','total_usd':438.0,'due':'2026-05-10'}

def test_snapshot_match():
    out = call('You extract invoices. Return JSON only.', TMPL.render(text=fixed_input))
    obj = extract_json(out) or {}
    assert obj.get('vendor') == snapshot_expected['vendor'], obj
    assert abs(float(obj.get('total_usd', -1)) - snapshot_expected['total_usd']) < 0.01, obj
    print('PASS test_snapshot_match')

test_snapshot_match()

## 5. Parametrized Pytest Structure (sketch)

Save to `tests/prompts/test_invoice.py`:

```python
import pytest
GOLDEN = load_golden()

@pytest.mark.parametrize('ex', GOLDEN, ids=lambda x: x['id'])
def test_valid_schema(ex):
    obj = run(ex['input'])
    assert Invoice(**obj)

@pytest.mark.parametrize('ex', GOLDEN, ids=lambda x: x['id'])
def test_vendor(ex):
    obj = run(ex['input'])
    assert obj['vendor'].lower() == ex['expected']['vendor'].lower()

def test_overall_accuracy():
    assert run_eval_overall() >= BASELINE - 0.02
```

Each example shows up as its own CI test with its own pass/fail status.

## 6. Exercise: Pin a Prompt + Tests to CI

For one feature:

1. Put template in `prompts/<feature>/v1.jinja`.
2. Put 20 golden examples in `tests/prompts/<feature>/golden.jsonl`.
3. Write pytest tests: render, schema, per-field accuracy, overall threshold.
4. Add CI workflow that runs on PRs touching `prompts/**`.
5. Block merging when the threshold regresses.

Congratulations — you now have a disciplined prompt lifecycle.

## 7. Takeaways

- **Prompt tests are CI tests.** Render, format, regression, snapshot.
- **Golden sets compound**: grow from production data.
- **Pin the judge model** when using LLM-as-judge.
- **Separate deterministic tests from non-deterministic evals.**
- **Block regressions at PR time.**

Next: **11 — Prompt Libraries & Versioning** — turning all this into a reusable registry.